# 05. Online-inference request-rate–power gray-box model

This notebook builds the first gray-box component: a steady-state map from NLR request rate to measured IT active power. It compares three physically interpretable monotone saturation structures without treating the two prompt datasets as repeated measurements.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'src').is_dir() else cwd.parent
FEATURE_PATH = PROJECT_ROOT / 'outputs' / 'feature_analysis' / 'analysis_ready_features.parquet'
AUDIT_PATH = PROJECT_ROOT / 'outputs' / 'condition_audit' / 'audited_run_manifest.parquet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'graybox_steady_state'
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from calibration.steady_state_calibration import run_steady_state_calibration

result = run_steady_state_calibration(FEATURE_PATH, AUDIT_PATH, OUTPUT_DIR)
tables = result['tables']
targets = tables['online_rate_condition_targets']
metrics = tables['steady_state_metrics']
predictions = tables['steady_state_predictions']
parameters = tables['steady_state_parameters']
best = tables['best_model_by_holdout']
print(f'Target cases: {len(targets):,}')
print(f'Unique request rates: {targets.request_rate.nunique():,}')
print(f'Prompt datasets: {targets.prompt_dataset.nunique():,}')

## 1. Calibration target and system boundary

The target is the mean IT active power within each sustained three-minute online-rate trace. NLR does not provide repeated traces for these strict conditions, so the within-trace standard deviation is retained only as a temporal-variation descriptor—not as run-to-run uncertainty or a fitting weight. Cooling, UPS losses, TES, and PCC power are outside this first model boundary.

In [ ]:
coverage = targets.groupby('prompt_dataset').agg(
    cases=('profile_id', 'size'),
    unique_rates=('request_rate', 'nunique'),
    min_rate=('request_rate', 'min'),
    max_rate=('request_rate', 'max'),
    min_power_kw=('target_power_kw', 'min'),
    max_power_kw=('target_power_kw', 'max'),
).reset_index()
display(coverage)
display(targets.head())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for dataset, group in targets.groupby('prompt_dataset'):
    group = group.sort_values('request_rate')
    ax.plot(group.request_rate, group.target_power_kw, marker='o', markersize=3, linewidth=1.2, label=dataset)
ax.set(xlabel='Request rate (request/s)', ylabel='Mean measured IT power (kW)', title='NLR sustained online-inference operating points')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'online_rate_measured_targets.png', dpi=180, bbox_inches='tight')
plt.show()

## 2. Candidate mathematical structures

All candidates enforce nonnegative, monotone, saturating power. The equations are research candidates synthesized from the observed saturation behavior and gray-box modeling principles; the NLR paper itself does not prescribe one exact saturation equation.

In [ ]:
display(tables['steady_state_model_registry'])

## 3. Leakage-safe evaluation design

Interpolation holds out regularly spaced interior request rates. High-rate extrapolation holds out the highest 20% of request rates. Both prompt datasets at a given request rate always share one partition, preventing request-rate leakage in the pooled model.

In [ ]:
split_summary = tables['request_rate_split_assignments'].groupby(['split_scheme', 'partition']).agg(
    rate_count=('request_rate', 'nunique'),
    min_rate=('request_rate', 'min'),
    max_rate=('request_rate', 'max'),
).reset_index()
display(split_summary)

## 4. Held-out model comparison

Held-out RMSE is the primary structure-selection metric. MAPE and high-rate saturation bias are reported alongside it; AIC/BIC are secondary because predictive generalization is the main purpose of the split.

In [ ]:
comparison = metrics.query("evaluation_partition == 'test'").copy()
display(comparison[[
    'scope', 'split_scheme', 'model_name', 'sample_count', 'rmse_kw', 'mae_kw',
    'mape_pct', 'r2', 'high_rate_bias_kw', 'high_rate_saturation_error_pct',
    'monotonicity_violation_count'
]].sort_values(['scope', 'split_scheme', 'rmse_kw']))
print('Best model in each held-out scenario:')
display(best[[
    'scope', 'split_scheme', 'model_name', 'rmse_kw', 'mape_pct', 'r2', 'high_rate_bias_kw'
]])

In [ ]:
pooled_comparison = comparison.query("scope == 'pooled'").copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (scheme, group) in zip(axes, pooled_comparison.groupby('split_scheme', sort=True)):
    group = group.sort_values('rmse_kw')
    ax.bar(group.model_name, group.rmse_kw)
    ax.set_title(scheme.replace('_', ' '))
    ax.set_ylabel('Held-out RMSE (kW)')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.25)
fig.suptitle('Pooled-model held-out comparison')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pooled_holdout_rmse.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Full-data calibration for the next modeling stage

After selecting a structure using held-out cases, its final parameters are refitted on all 100 request rates. These full-fit curves become the steady-state block used later in the finite-workload ramp/dynamic model.

In [ ]:
full_predictions = predictions.query("scope == 'pooled' and split_scheme == 'full_fit'").copy()
fig, ax = plt.subplots(figsize=(9, 5))
for dataset, group in targets.groupby('prompt_dataset'):
    ax.scatter(group.request_rate, group.target_power_kw, s=14, alpha=0.65, label=f'observed: {dataset}')
for model_name, group in full_predictions.groupby('model_name'):
    curve = group.groupby('request_rate', as_index=False).predicted_power_kw.mean().sort_values('request_rate')
    ax.plot(curve.request_rate, curve.predicted_power_kw, linewidth=2, label=f'model: {model_name}')
ax.set(xlabel='Request rate (request/s)', ylabel='IT power (kW)', title='Full-data pooled gray-box calibration')
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'pooled_full_fit_curves.png', dpi=180, bbox_inches='tight')
plt.show()

full_parameter_table = parameters.query("scope == 'pooled' and split_scheme == 'full_fit'").pivot(
    index='model_name', columns='parameter', values='value'
)
display(full_parameter_table)

## Interpretation and next step

This stage selects and calibrates only the sustained request-rate–mean-power map. It does not yet constitute the complete AIDC gray-box model. The next notebook should use `online_inference_finite` traces to identify startup/shutdown behavior and a first-order time constant, after which stochastic residual and burst components can be added and validated against profile-level ramp and temporal statistics.